In [2]:
import csv
import time
from bs4 import BeautifulSoup # parseo de html 
from selenium import webdriver # abrir chrome
from selenium.webdriver.chrome.options import Options as ChromeOptions # configurarlo
from selenium.webdriver.chrome.service import Service # conectar chromedriver
from selenium.webdriver.support.ui import WebDriverWait # esperar que cargue la pagina
from selenium.webdriver.support import expected_conditions as EC # define que espera
from selenium.webdriver.common.by import By # para indicar como buscar elem
from webdriver_manager.chrome import ChromeDriverManager # para descargar el chrome driver directo

COMUNAS = [
    ("Valdivia", "valdivia-de-los-rios"),
    ("Mariquina", "mariquina-de-los-rios"),
    ("Lanco", "lanco-de-los-rios"),
    ("Los Lagos", "los-lagos-de-los-rios"),
    ("Máfil", "mafil-de-los-rios"),
    ("Corral", "corral-de-los-rios"),
    ("Futrono", "futrono-de-los-rios"),
    ("La Unión", "la-union-de-los-rios"),
    ("Lago Ranco", "lago-ranco-de-los-rios"),
    ("Río Bueno", "rio-bueno-de-los-rios"),
    ("Panguipulli", "panguipulli-de-los-rios"),
    ("Paillaco", "paillaco-de-los-rios"),
]
 
MODALIDADES = [
    ("arriendo", "arriendo"),
    ("venta", "venta"),
]
 
TIPOS = [
    ("casa", "casa"),
    ("departamento", "departamento"),
]
 
ARCHIVO_CSV  = "propiedades_los_rios.csv"
PAUSA        = 2
AVISOS_X_PAG = 48
VALOR_UF = 40000
# abrir chrome 
def crear_driver():
    chrome_options = ChromeOptions()
    chrome_options.add_argument("--headless=new") # ventana
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
 
def extraer_propiedades(soup, tipo, modalidad, comuna):
    propiedades = []
    avisos = soup.find_all("li", class_="ui-search-layout__item") # busco cada aviso ( cuadrado con datos)
 
    for aviso in avisos:
        # Precio
        precio = "N/D"
        fraccion = aviso.find("span", class_="andes-money-amount__fraction")
        if fraccion:
            moneda_tag = aviso.find("span", class_="andes-money-amount__currency-symbol")
            moneda = moneda_tag.get_text(strip=True) if moneda_tag else "$"
            texto = fraccion.get_text(strip=True).replace(".", "").replace(",", ".")
            if moneda == "UF":
                precio = str(int(float(texto) * VALOR_UF))
            else:
                precio = fraccion.get_text(strip=True)

 
        # m2 utiles
        # hay varios atributos por eso busco y si tiene m2 util extrae y lo anota
        m2_utiles = "N/D"
        atributos = aviso.find_all("li", class_="poly-attributes_list__item")
        for atrib in atributos:
            texto = atrib.get_text(strip=True)
            if "m²" in texto and "útil" in texto:
                m2_utiles = texto
                break
 
        # ubicación
        ubicacion = "N/D"
        ub_tag = aviso.find("span", class_="poly-component__location")
        if ub_tag:
            ubicacion = ub_tag.get_text(strip=True)
 
        propiedades.append({
            "tipo": tipo,
            "modalidad": modalidad,
            "comuna": comuna,
            "precio": precio,
            "m2_utiles": m2_utiles,
            "ubicacion": ubicacion,
        }) # guardo en diccionario
 
    return propiedades
 
# scraping principal
 
def scrapear():
    driver   = crear_driver()
    todas    = []
    id_aviso = 1
    print("Empezando")
 
 
    try:
        for nombre_tipo, slug_tipo in TIPOS:
            for nombre_modalidad, slug_modalidad in MODALIDADES:
                for nombre_comuna, slug_comuna in COMUNAS:
 
                    url_base = f"https://www.portalinmobiliario.com/{slug_modalidad}/{slug_tipo}/{slug_comuna}"
                    offset   = 1
                    pagina   = 1
 
                    print(f"\n {nombre_tipo} | {nombre_modalidad} | {nombre_comuna}")
 
                    while True:
                        url = url_base if offset == 1 else f"{url_base}/_Desde_{offset}" # para avanzar en pagina
 
                        print(f"  Página {pagina}...", end=" ", flush=True)
                        driver.get(url)
                        #sirve para ver si hay avisos, sino break y pasa al siguiente
                        try:
                            WebDriverWait(driver, 15).until(
                                EC.presence_of_element_located(
                                    (By.CLASS_NAME, "ui-search-layout__item")
                                )
                            )
                        except Exception:
                            print("sin avisos, fin.")
                            break

                        # toma el html y extrae
                        soup  = BeautifulSoup(driver.page_source, "lxml")
                        props = extraer_propiedades(soup, nombre_tipo, nombre_modalidad, nombre_comuna)
                        print(f"{len(props)} avisos")
 
                        if not props:
                            break
 
                        for prop in props:
                            prop["id"] = id_aviso
                            id_aviso  += 1
                        # asigna id
                        #agrega a todas
                        todas.extend(props)
                        # si entrega menos de 48 avisos es la ultima pag
                        if len(props) < AVISOS_X_PAG:
                            break
                        # avanza
                        offset += AVISOS_X_PAG
                        pagina += 1
                        time.sleep(PAUSA)
 
    finally:
        driver.quit()
 
    # guardar csv
    campos = ["id", "tipo", "modalidad", "comuna", "precio", "m2_utiles", "ubicacion"]
    with open(ARCHIVO_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=campos)
        writer.writeheader()
        writer.writerows(todas)
 

    print(f"\n {len(todas)} avisos guardados en '{ARCHIVO_CSV}'")
 
 

scrapear()
    

Empezando

 casa | arriendo | Valdivia
  Página 1... 48 avisos
  Página 2... 48 avisos
  Página 3... 11 avisos

 casa | arriendo | Mariquina
  Página 1... sin avisos, fin.

 casa | arriendo | Lanco
  Página 1... 1 avisos

 casa | arriendo | Los Lagos
  Página 1... 2 avisos

 casa | arriendo | Máfil
  Página 1... 1 avisos

 casa | arriendo | Corral
  Página 1... 1 avisos

 casa | arriendo | Futrono
  Página 1... 18 avisos

 casa | arriendo | La Unión
  Página 1... 1 avisos

 casa | arriendo | Lago Ranco
  Página 1... sin avisos, fin.

 casa | arriendo | Río Bueno
  Página 1... 4 avisos

 casa | arriendo | Panguipulli
  Página 1... 12 avisos

 casa | arriendo | Paillaco
  Página 1... 1 avisos

 casa | venta | Valdivia
  Página 1... 48 avisos
  Página 2... 48 avisos
  Página 3... 48 avisos
  Página 4... 48 avisos
  Página 5... 48 avisos
  Página 6... 48 avisos
  Página 7... 48 avisos
  Página 8... 48 avisos
  Página 9... 48 avisos
  Página 10... 48 avisos
  Página 11... 48 avisos
  Página